### Link to article

This notebook is included in the documentation, where the interactive Plotly charts show up. See:
https://pegasystems.github.io/pega-datascientist-tools/latest/articles/Example_ADM_Analysis.html

> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.
>
> Generating the reports at the end also needs the `healthcheck` extra, plus [Quarto](https://quarto.org/) and [Pandoc](https://pandoc.org/).

# Example ADM analysis
See this notebook for an introduction to the ADMDatamart class to get an overview of the currently implemented features in the Python version of CDH Tools. If you have any suggestions for new features, please do not hesitate to raise an issue in Git, or even better: create a pull request yourself!

### Reading the data
Use `ADMDatamart.from_ds_export(base_path="...")` to import the latest snapshots from a directory. For this example, we generate data from the shared synthetic demo world:

In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately
# Hidden from doc by virtue of cell tags - in VSCode right-click on the bar to the left of this cell, edit cell tags, see metadata

import plotly.io as pio

pio.renderers.default = "notebook_connected"


In [2]:
from pdstools import demo_data
import polars as pl

# Notebook examples intentionally generate data; the Health Check app uses source="snapshot".
dm = demo_data(source="generated").datamart


### Bubble Chart
To start out with the bubble chart, which we can simply call by calling [`plot.bubble_chart`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots) on our main class.

In [3]:
fig = dm.plot.bubble_chart()
fig.show()

Looks like a healthy bubble plot, but sometimes it is useful to consider only certain models in the analysis. Note that the bubble chart automatically considers only the last snapshot by default, though this is a parameter. 

To reduce the information, let's only consider models with more than 500 responses within the Loans group. 

In [4]:
query = (pl.col("ResponseCount") > 500) & (pl.col("Group") == "Loans")
fig = dm.plot.bubble_chart(query=query)
fig.show()

Alternatively, we could only look at the top n best performing models within our query. To do this, we need to supply a list of model IDs which we can easily extract from the data as such.

Note here the alternative querying syntax you can use, which was default in the previous version of CDH Tools: if you have a list (list) to subset a column's values with, you can simply supply a dictionary with 'column name':list to only get values in that list for that column.

In [5]:
top30ids = (
    dm.aggregates.last()
    .sort("Performance", descending=True)
    .select("ModelID")
    .head(30)
    .collect()
    .to_series()
    .to_list()
)

fig = dm.plot.bubble_chart(query={"ModelID": top30ids})
fig.show()

The bubble chart gives some information about which models perform well, but that is not always informative: if we don't know in which channels, issues or groups our issues lie then we may not be looking in the right place. This is where the Treemap visualisation is quite handy. 

In [6]:
fig = dm.plot.tree_map()
fig.show()

By default the Treemap shows the weighted performance, where the performance is weighted by the response count. The squares represent Model IDs: the larger a square, the more model IDs are within that combination of context keys. We can also color the Treemap by another variable, such as the SuccessRate:

In [7]:
fig = dm.plot.tree_map("SuccessRate")
fig.show()

`plot.over_time` can also show the success rate over time. By default, it shows the cumulative success rate. Set `cumulative=False` to show the success rate within each interval, from the changes in the cumulative positives and responses of each model. `every` takes a Polars interval such as `"1d"` or `"1w"`; `every="auto"` picks daily, weekly or monthly intervals from the date range. Intervals with a 100% success rate are marked with a red X; the y-axis is scaled to the other values.

In [8]:
fig = dm.plot.over_time(
    "SuccessRate",
    by="ModelID",
    query=pl.col("Channel") == "Web",
    every="auto",
    cumulative=False,
)
fig.show()

over_time: 350 unique values for `ModelID` exceeds the cap of 10; keeping the top 10 by total ResponseCount and dropping 340.


And if it is not interesting to consider the success rate over time, there is also [`plot.proposition_success_rates`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots), which by default considers the last state of the models and plots the histogram of their success rates.

In [9]:
fig = dm.plot.proposition_success_rates(query=pl.col("Channel") == "Web")
fig.show()

A low success rate on a model with few responses says little. [`plot.success_rate_intervals`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots) puts every model at its number of negatives and positives (logarithmic axes, so lines through the origin are constant success rates) and adds the confidence interval of its success rate. There is one panel per channel and direction, each with its own average (the dashed line). Models are coloured by whether the interval is entirely below, entirely above, or consistent with that average.

In [10]:
fig = dm.plot.success_rate_intervals()
fig.show()

If we want to look at the distribution of responses and their propensities for a given model, we can subset that model and call [`plot.score_distribution`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots). Note here we subset the model by its ID.

In [11]:
fig = dm.plot.score_distribution(model_id="05085857-c52c-5daa-b51b-0af576a038af")
fig.show()

Alternatively, we can also subset a model by its model name, and then further drill down by group/issue/channel/configuration. See the example below.

In [12]:
figs = dm.plot.multiple_score_distributions(
    query=(pl.col("Name") == "FixedRateLoan")
    & (pl.col("Group") == "Loans")
    & (pl.col("Issue") == "Sales")
    & (pl.col("Channel") == "Web")
    & (pl.col("Configuration") == "OmniAdaptiveModel"),
    show_all=False,
)
for fig in figs:
    fig.show()

Similarly, we can also display the distribution of a single predictor and its binning. This function loops through each predictor of a model and generates the binning image for that predictor. For that reason we recommend subsetting the predictor names ahead of time or, depending on how many predictors the model has, a lot of images will be generated.

In [13]:
figs = dm.plot.multiple_predictor_binning(
    model_id="05085857-c52c-5daa-b51b-0af576a038af",
    query=(
        pl.col("PredictorName").is_in(
            [
                "Customer.HomePageVisitsLast30Days",
                "IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince",
                "Param.MessageLengthBand",
            ]
        )
    ),
    show_all=False,
);
for fig in figs:
    fig.show()

Alternatively we can look at the performance of a predictor over multiple models. Again, we recommend subsetting the predictor names with a list to make it more legible. 

In [14]:
fig = dm.plot.predictor_performance(
    query=pl.col("PredictorName").is_in(
        [
            "Customer.HomePageVisitsLast30Days",
            "IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince",
            "Param.MessageLengthBand",
        ]
    )
)
fig.show()

What the two previous visualisations could not represent very well is the performance of the predictors over different models. That is what [`plot.predictor_performance_heatmap`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots) does; again with subsetting of predictors as a recommended step.

In [15]:
fig = dm.plot.predictor_performance_heatmap(
    query=pl.col("PredictorName").is_in(
        [
            "Customer.HomePageVisitsLast30Days",
            "IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince",
            "Param.MessageLengthBand",
        ]
    )
)
fig.show()

### External model scores as ADM predictors

When you integrate scores from third-party ML models or custom scoring pipelines
as ADM predictors, you may want to analyse their contribution separately from
native customer-profile or interaction-history predictors.

The synthetic demo includes `Customer.ChurnRiskModelScore`, a precomputed
score automatically classified as `External Model` by its `ModelScore`
suffix. It is not included in the restricted Web Loans configuration, and
no external model is downloaded or run.

For other naming conventions, [`apply_predictor_categorization`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/ADMDatamart/index.html#pdstools.adm.ADMDatamart.ADMDatamart.apply_predictor_categorization)
accepts a dictionary of category labels and name substrings, or regular
expressions with `use_regexp=True`. The mapping below matches model scores
and two example external-score prefixes without relabeling ordinary customer
scores or interaction-history predictors.

In [16]:
dm.apply_predictor_categorization(
    categorization={"External Model": [r"(?i)^Customer\..*model_?score$", r"^(ExtScore_|ModelOutput\.)"]},
    use_regexp=True,
)

#### Naive Bayes: univariate predictor performance

[`plot.predictor_performance`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots) shows univariate performance across
the selected Mobile Naive Bayes models. The Health Check eligibility and
response-count filters keep the comparison consistent with the report.

In [17]:
mobile_nb = (
    (pl.col("Configuration") == "Mobile_Click_Through_Rate")
    & (pl.col("Channel") == "Mobile")
    & (pl.col("ResponseCountBin") > 200)
    & dm.analysis.health_check_active_filter()
)
nb_fig = dm.plot.predictor_performance(query=mobile_nb, top_n=30)
nb_fig.show()

#### Adaptive Gradient Boosting: feature importance

AGB does not provide Naive Bayes-style univariate performance. Use
`plot.predictor_importance` to compare the average model importance of the
same external score with other predictors in the Mobile AGB configuration.

In [18]:
mobile_agb = (
    (pl.col("Configuration") == "Mobile_Click_Through_Rate_GB")
    & (pl.col("Channel") == "Mobile")
    & dm.analysis.health_check_active_filter()
)
agb_fig = dm.plot.predictor_importance(query=mobile_agb, top_n=20)
agb_fig.show()

#### Contribution by predictor category

[`plot.predictor_contribution`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/adm/Plots/index.html#pdstools.adm.Plots.Plots) compares response-weighted average
univariate performance by category, normalized within each configuration.
Here we limit the plot to Naive Bayes, where univariate performance is available.

In [19]:
fig = dm.plot.predictor_contribution(query=pl.col("ModelTechnique") == "NaiveBayes")
fig.show()

#### Interpreting the results

**Predictor performance** measures how well a single predictor separates
positives from negatives within a Naive Bayes model (AUC on the 50–100 scale).
A higher value indicates stronger univariate separation, not necessarily a
causal effect or the predictor's independent contribution to the model.

**AGB importance** compares the model-reported importance of predictors
within the selected models. **Contribution** rescales univariate AUC above
0.5, computes a response-weighted mean for each category, then normalizes
the category means to percentages. Those percentages are descriptive; they
are not an additive or causal share of the model's predictive power.

Typical action items:

* **Strong separation or importance** — check score freshness and fallback
  behavior for missing values.
* **A large relative category bar but modest absolute performance** — compare
  the underlying AUCs before drawing conclusions.
* **A low or missing score** — inspect activation, coverage and data
  availability; a category percentage alone cannot establish whether a model
  uses the score.

## Generating reports

Use these examples to create an ADM Health Check or an individual Naive Bayes Model Report from code. They use the reproducible synthetic sample; for your own data, use your own `ADMDatamart` instance and model ID, and pass a matching `Prediction` object to include Prediction Studio analysis in a Health Check.

The ADM Health Check Streamlit app wraps these report generators for interactive use. Run `pdstools` and select **Adaptive Model Health Check** to launch it.

### ADM Health Check report

Generate the Health Check, including the sample's Prediction Studio data:

```python
from pdstools import demo_data

# Notebook examples intentionally generate data; the Health Check app uses source="snapshot".
sample = demo_data(source="generated")
sample.datamart.generate.health_check(
    prediction=sample.prediction,
    output_dir="reports",
    full_embed=False,
)
```

### Naive Bayes Model Report

Generate a report for the sample Savings Review model for Mobile:

```python
from pdstools import demo_data

# Notebook examples intentionally generate data; the Health Check app uses source="snapshot".
sample = demo_data(source="generated")
sample.datamart.generate.model_reports(
    "bd72621c-78af-5a37-997b-0a7f4d214132",
    output_dir="reports",
    full_embed=False,
)
```